# 실습 2. 두 군의 크기 비교

사회약학 연구방법 노트의 '실습 2. 두 군의 크기 비교'에 딸린 노트북입니다. MASS 패키지의 birthwt 자료(1986년 미국 한 병원의 출생아 189명)로 임신 중 흡연한 어머니와 흡연하지 않은 어머니의 출생아 체중을 Student·Welch t 검정과 Mann-Whitney 검정으로 비교하고, 평균 차이의 신뢰구간, Cohen의 d, 우월 확률, Hodges–Lehmann 추정치를 구합니다. 셀을 위에서부터 차례로 실행하세요. 설명과 출력 읽는 법은 사이트의 실습 2에 있습니다.


## a. 실습 데이터 준비

이 실습이 쓰는 패키지를 불러와 버전을 확인한 뒤 birthwt 자료를 읽습니다. 자료는 인터넷에서 읽어 오므로 인터넷 연결이 필요합니다.

### 셀 1. 패키지 불러오기와 버전 확인

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import scipy, matplotlib, seaborn, sklearn

print("pandas      ", pd.__version__)
print("numpy       ", np.__version__)
print("scipy       ", scipy.__version__)
print("matplotlib  ", matplotlib.__version__)
print("seaborn     ", seaborn.__version__)
print("scikit-learn", sklearn.__version__)

### 셀 2. 데이터 불러오기

In [ ]:
url = ("https://vincentarelbundock.github.io/Rdatasets/"
       "csv/MASS/birthwt.csv")
df = pd.read_csv(url)
print(df.shape)
df.head()

### 셀 3. 흡연 여부에 이름표 붙이기

In [ ]:
df["smoke_f"] = pd.Categorical(
    df["smoke"].map({0: "Nonsmoker", 1: "Smoker"}),
    categories=["Nonsmoker", "Smoker"])
df["smoke_f"].value_counts(sort=False)

### 셀 4. 군별 요약통계

In [ ]:
summary = (df.groupby("smoke_f", observed=True)["bwt"]
             .agg(["count", "mean", "std", "median"]))
summary.round(1)

열 이름을 잘못 써서 일부러 오류를 내 보는 셀(셀 5)은 이 노트북에서 뺐습니다(사이트의 실습 2 a 절에 있습니다).

### 셀 6. 상자그림과 개별 점

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, ax = plt.subplots(figsize=(5, 4))
sns.boxplot(data=df, x="smoke_f", y="bwt", ax=ax,
            color="0.85", width=0.5, showfliers=False)
np.random.seed(0)        # 점 흩뿌리기(jitter) 위치 고정
sns.stripplot(data=df, x="smoke_f", y="bwt", ax=ax,
              color="black", alpha=0.4, jitter=0.2)
ax.axhline(2500, ls="--", color="gray")
ax.set_xlabel("Smoking during pregnancy")
ax.set_ylabel("Birth weight (g)")
plt.show()

## b. 독립표본 t 검정

### 셀 7. Student t 검정과 Welch t 검정

In [ ]:
smoker = df.loc[df["smoke"] == 1, "bwt"]
nonsmoker = df.loc[df["smoke"] == 0, "bwt"]

student = stats.ttest_ind(smoker, nonsmoker, equal_var=True)
welch = stats.ttest_ind(smoker, nonsmoker, equal_var=False)
print(welch)
print(f"Student: t = {student.statistic:.3f}, "
      f"p = {student.pvalue:.4f}, df = {student.df:.0f}")
print(f"Welch:   t = {welch.statistic:.3f}, "
      f"p = {welch.pvalue:.4f}, df = {welch.df:.1f}")

### 셀 8. 평균 차이와 95% 신뢰구간

In [ ]:
diff = smoker.mean() - nonsmoker.mean()
ci = welch.confidence_interval(confidence_level=0.95)
print(f"평균 차이 = {diff:.1f} g")
print(f"95% CI = {ci.low:.1f} ~ {ci.high:.1f} g")

### 셀 9. 같은 값을 공식으로 계산하기

In [ ]:
n1, n2 = len(smoker), len(nonsmoker)
v1, v2 = smoker.var(), nonsmoker.var()   # 분산 (n - 1로 나눔)
se = np.sqrt(v1 / n1 + v2 / n2)          # 차이의 표준오차
df_w = (v1/n1 + v2/n2)**2 / (
    (v1/n1)**2 / (n1 - 1) + (v2/n2)**2 / (n2 - 1))
t_crit = stats.t.ppf(0.975, df_w)        # 양측 5% 임계값
print(f"SE = {se:.2f}, t = {diff / se:.3f}, df = {df_w:.2f}")
print(f"CI = {diff - t_crit*se:.1f} ~ {diff + t_crit*se:.1f}")

### 셀 10. Cohen의 d

In [ ]:
sp = np.sqrt(((n1 - 1) * v1 + (n2 - 1) * v2) / (n1 + n2 - 2))
d = diff / sp
print(f"합동 SD = {sp:.1f} g")
print(f"Cohen's d = {d:.2f}")

### 셀 11. Levene 검정

In [ ]:
lev_med = stats.levene(smoker, nonsmoker)   # 중앙값 기준 (기본값)
lev_mean = stats.levene(smoker, nonsmoker, center="mean")
print(lev_med)
print(f"median: p = {lev_med.pvalue:.3f}")
print(f"mean:   p = {lev_mean.pvalue:.3f}")

## c. Mann-Whitney 검정

### 셀 12. 중앙값과 사분위수

In [ ]:
q = (df.groupby("smoke_f", observed=True)["bwt"]
       .quantile([0.25, 0.5, 0.75])
       .unstack())
q

### 셀 13. Mann-Whitney 검정

In [ ]:
mw = stats.mannwhitneyu(smoker, nonsmoker,
                        alternative="two-sided")
print(mw)
n1n2 = n1 * n2
print("U(smoker) =", mw.statistic)
print("U(nonsmoker) =", n1n2 - mw.statistic)
print(f"p = {mw.pvalue:.4f}")

### 셀 14. U를 직접 세어 보기

In [ ]:
s = smoker.to_numpy()[:, None]      # 74행 1열
ns = nonsmoker.to_numpy()[None, :]  # 1행 115열
greater = (s > ns).sum()            # 흡연군 쪽이 더 무거운 쌍
ties = (s == ns).sum()              # 체중이 같은 쌍
print(greater, ties, greater + 0.5 * ties)

### 셀 15. 우월 확률과 AUC

In [ ]:
ps = mw.statistic / n1n2
print(f"P(smoker > nonsmoker) = {ps:.3f}")

from sklearn.metrics import roc_auc_score
auc = roc_auc_score(df["smoke"], df["bwt"])
print(f"AUC = {auc:.3f}")

### 셀 16. Hodges-Lehmann 추정치

In [ ]:
diffs = np.subtract.outer(smoker.to_numpy(),
                          nonsmoker.to_numpy())
print(diffs.shape)
hl = np.median(diffs)

d_sorted = np.sort(diffs.ravel())
N = n1 + n2
k = round(n1n2 / 2 - 1.96 * np.sqrt(n1n2 * (N + 1) / 12))
lo, hi = d_sorted[k - 1], d_sorted[n1n2 - k]
print(f"HL = {hl:.1f} g, 95% CI {lo} ~ {hi}")

## d. 과제 정답

**과제 1.** 셀 8의 평균 차이와 95% 신뢰구간을 논문의 Results처럼 한 문장으로 옮기고, 이 숫자가 말해 주지 않는 것을 두 가지 이상 쓰세요. 비흡연군 평균보다 무거운 흡연군 아기가 몇 명인지, 인종별 흡연 비율과 출생체중도 확인해 보세요.

### 셀 17. 과제 1 정답. 평균 차이가 말해 주지 않는 것

In [ ]:
above = (smoker > nonsmoker.mean()).sum()      # 비흡연군 평균보다 무거운 흡연군 아기
print("smokers' infants above the nonsmoker mean:", above, "of", n1)

df["race_f"] = pd.Categorical(
    df["race"].map({1: "white", 2: "black", 3: "other"}),
    categories=["white", "black", "other"])
print(pd.crosstab(df["race_f"], df["smoke_f"], normalize="index").round(3))
df.groupby(["race_f", "smoke_f"], observed=True)["bwt"].mean().unstack().round(1)

**과제 2.** 셀 7의 두 t 검정을 흡연 대신 고혈압 병력(`ht`)으로 두 군을 나눠 다시 하세요. Student와 Welch의 P와 95% 신뢰구간이 어떻게 다른지, 흡연에서는 비슷했던 두 결과가 왜 여기서는 크게 갈리는지 두 군의 인원과 SD로 설명하세요.

### 셀 18. 과제 2 정답. 고혈압 병력으로 나눈 두 군의 Student와 Welch

In [ ]:
ht1 = df.loc[df["ht"] == 1, "bwt"]               # 고혈압 병력 있음
ht0 = df.loc[df["ht"] == 0, "bwt"]               # 없음
print(f"n = {len(ht1)} vs {len(ht0)}, "
      f"SD = {ht1.std():.1f} vs {ht0.std():.1f}")
for name, ev in [("Student", True), ("Welch", False)]:
    r = stats.ttest_ind(ht1, ht0, equal_var=ev)
    ci_ht = r.confidence_interval()
    print(f"{name:7}: diff = {ht1.mean() - ht0.mean():.1f}, "
          f"t = {r.statistic:.2f}, df = {r.df:.1f}, p = {r.pvalue:.3f}, "
          f"95% CI {ci_ht.low:.0f} ~ {ci_ht.high:.0f}")

**과제 3.** 동료가 결과를 "흡연군 아기의 출생체중은 2,772 ± 660 g, 비흡연군은 3,056 ± 753 g였고, Mann-Whitney 검정에서 흡연군의 중앙값이 324.5 g 유의하게 낮았다(P = 0.007)"라고 썼습니다. 틀린 곳 두 가지를 찾아 설명하세요. 흡연군의 위쪽 4분의 1 값만 500 g 무겁게 바꾸면 중앙값의 차이와 P가 어떻게 되는지 확인해 보세요.

### 셀 19. 과제 3 정답. Mann-Whitney 검정의 P는 중앙값의 차이를 검정하지 않습니다

In [ ]:
print("median difference:", smoker.median() - nonsmoker.median())
q75 = smoker.quantile(0.75)
smoker_hi = smoker.where(smoker <= q75, smoker + 500)  # 위쪽 4분의 1만 +500 g
for name, x in [("original", smoker), ("top quarter +500", smoker_hi)]:
    r = stats.mannwhitneyu(x, nonsmoker, alternative="two-sided")
    print(f"{name:16}: median diff = {x.median() - nonsmoker.median():.1f}, "
          f"P(smoker > nonsmoker) = {r.statistic / n1n2:.3f}, "
          f"p = {r.pvalue:.4f}")
print(f"Hodges-Lehmann = {hl:.1f} g, 95% CI {lo} ~ {hi}")